# Structured Extraction -- Streamlining Auto-Insurance Claims Intake

**High-volume operational documents, where manual data entry is the bottleneck.** A constant stream of claim documents lands in one place -- first-notice-of-loss forms, repair estimates, police reports, and photos of vehicle damage, all mixed together, in several insurer layouts.

This pipeline turns that pile into decisions. It:

1. **Classifies** every document by type -- the *traffic controller* (`AI_CLASSIFY`).
2. **Routes** each type to its own extractor, with per-field **confidence scores**.
3. **Assesses the damage photos** with vision as a structured field.
4. **Assembles** one record per claim across all of its documents.
5. **Decides** -- a recommended action, a fraud-risk flag, and a settlement estimate.
6. **Triages** into `auto_settle` / `needs_review` / `reject` lanes -- anything the model is unsure about goes to a human.

Everything is declarative Snowflake Cortex SQL on incremental dynamic tables: each AI function runs **once per new document**, not once per refresh.

```
DEMO_CLM_DOCS_STAGE (mixed PDFs + photos in incoming/)
  -> DEMO_CLM_FILE_LOG          stream + task (event-driven ingest)
  -> DT_DEMO_CLM_CLASSIFIED     AI_CLASSIFY(TO_FILE) -> DOC_TYPE     [traffic controller + junk gate]
  -> DT_DEMO_CLM_PARSED         AI_PARSE_DOCUMENT(LAYOUT)  (textual types only)
  -> routed extraction:
       DT_DEMO_CLM_FNOL         AI_EXTRACT(scores=>TRUE)  fields + confidence
       DT_DEMO_CLM_ESTIMATE     AI_EXTRACT                shop, total
       DT_DEMO_CLM_POLICE       AI_EXTRACT                report_no, fault, narrative
       DT_DEMO_CLM_PHOTO        AI_COMPLETE(vision+JSON)  severity, parts, fraud cues
  -> DT_DEMO_CLM_CLAIM          LEFT JOIN the four on CLAIM_NO -> one record per claim
  -> DT_DEMO_CLM_DECISION       AI_COMPLETE(json)  action, fraud_risk, settlement
  -> DT_DEMO_CLM_TRIAGED        confidence + decision -> ROUTE       [terminal]
  -> views DEMO_CLM_AUTO_SETTLE / _NEEDS_REVIEW / _REJECTED / _CLAIM_INTELLIGENCE
```

`CLAIM_NO` is the shared key that ties a packet together -- derived from the staged path, so even the photo (which has no claim number on it) attaches to the right claim. Classification itself is honest AI: it reads the document bytes, never the filename.

> **Before running:** this notebook reads objects the pipeline already built, so first run `00_setup.sql`, the sourcing script (`source_structured_extraction.py`, which also backfills the file log and loads ground truth), `10_pipeline.sql` (creates the dynamic tables, zero-spend), and `20_triage.sql` section A (the cost-gated AI refresh) -- then let the dynamic tables settle. Substitute `{database}` / `{schema}` / `{warehouse}` in the context cell below; all other object references resolve against the schema it sets.

In [ ]:
USE SCHEMA {database}.{schema};
USE WAREHOUSE {warehouse};

## 1 - The intake -- what arrived, and what each document is

The first job is the *traffic controller*: one `AI_CLASSIFY` call per file, over a mixed folder of PDFs **and** images. Note the `other` bucket -- non-claim "junk" documents are gated out here, so they never reach an extractor.

In [ ]:
SELECT DOC_TYPE,
       COUNT(*)                                           AS documents,
       ROUND(100 * RATIO_TO_REPORT(COUNT(*)) OVER (), 1)  AS pct
FROM DT_DEMO_CLM_CLASSIFIED
GROUP BY DOC_TYPE
ORDER BY documents DESC;

Every dynamic table is **incremental** -- its `refresh_mode` is `INCREMENTAL`, so when new documents land only those documents are processed; the AI functions never re-run on unchanged files. This is what makes the same SQL that runs on 40 claims run on millions.

In [ ]:
SHOW DYNAMIC TABLES LIKE 'DT_DEMO_CLM%';
SELECT "name", "refresh_mode", "target_lag", "scheduling_state"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
ORDER BY "name";

## 2 - Routing -> assembly -- one record per claim

Each document type flows to its own extractor with its own schema, and the results are reassembled on `CLAIM_NO`. A single row below is built from up to four separate documents: the **FNOL** supplies claimant / amount, the **estimate** the repair total, the **police report** the fault, and the **damage photo** a vision-assessed severity. Claims missing a type (no police report, no photo) keep those columns `NULL` -- the `LEFT JOIN` handles partial packets.

In [ ]:
SELECT CLAIM_NO, CLAIMANT, DATE_OF_LOSS, VEHICLE,
       AMOUNT_CLAIMED, SHOP, ESTIMATE_TOTAL,
       FAULT, SEVERITY, PARTS_AFFECTED, REPAIR_VS_TOTAL,
       HAS_POLICE, HAS_PHOTO
FROM DEMO_CLM_CLAIM_INTELLIGENCE
ORDER BY CLAIM_NO
LIMIT 8;

## 3 - The decision -- a judgment the documents don't state

`AI_COMPLETE` reasons over the assembled record to produce a **recommended action**, a **fraud-risk** flag, a **settlement estimate**, and a one-line rationale -- weighing cross-document inconsistencies (a claimed amount far above the repair estimate, a photo that contradicts the claimed severity, missing evidence). The rationale is the adjuster-facing explanation.

In [ ]:
SELECT CLAIM_NO,
       ROUND(AMOUNT_CLAIMED)      AS claimed,
       ROUND(ESTIMATE_TOTAL)      AS estimate,
       ROUND(SETTLEMENT_ESTIMATE) AS settlement,
       FRAUD_RISK, RECOMMENDED_ACTION, ROUTE,
       RATIONALE
FROM DEMO_CLM_CLAIM_INTELLIGENCE
ORDER BY SETTLEMENT_ESTIMATE DESC
LIMIT 10;

## 4 - Triage lanes -- the operational payoff

Each claim is routed into a lane by combining the model's judgment with hard business rules: a confidence gate on the extracted fields, the fraud-risk flag, and a high-value / missing-evidence backstop. The **auto-settle rate** is the headline metric; the rest is the work an adjuster picks up.

In [ ]:
SELECT ROUTE,
       COUNT(*)                                           AS claims,
       ROUND(SUM(SETTLEMENT_ESTIMATE))                    AS total_settlement_usd,
       ROUND(AVG(SETTLEMENT_ESTIMATE))                    AS avg_settlement_usd,
       ROUND(100 * RATIO_TO_REPORT(COUNT(*)) OVER (), 1)  AS pct_of_claims
FROM DT_DEMO_CLM_TRIAGED
GROUP BY ROUTE
ORDER BY claims DESC;

The **needs-review queue**, highest settlement first -- what a human adjuster works through, with the model's reason and the extraction confidence for each claim.

In [ ]:
SELECT CLAIM_NO, CLAIMANT,
       ROUND(SETTLEMENT_ESTIMATE) AS settlement,
       FRAUD_RISK,
       ROUND(MIN_CONFIDENCE, 2)   AS min_field_conf,
       RATIONALE
FROM DEMO_CLM_NEEDS_REVIEW
LIMIT 12;

And the **rejected** lane -- claims the model flagged as high fraud-risk or recommended to deny, with the photo fraud cues and rationale that drove it.

In [ ]:
SELECT CLAIM_NO,
       ROUND(AMOUNT_CLAIMED) AS claimed,
       ROUND(ESTIMATE_TOTAL) AS estimate,
       PHOTO_FRAUD_CUES,
       RATIONALE
FROM DEMO_CLM_REJECTED;

## 5 - Did it work? AI vs. ground truth

The corpus is synthetic, so we planted the answers -- fraud cues and the true field values -- that the pipeline never saw. Two checks:

**Fraud catch** -- of the cues we planted, which lane did each land in? The two money-fraud types (an inflated claim, an inflated estimate) get caught; subtler cues (a police fault contradiction that looks like a normal at-fault claim) are the honest argument for keeping a human in the loop.

In [ ]:
SELECT gt.PLANTED_FRAUD,
       COUNT(*)                              AS planted,
       COUNT_IF(t.ROUTE = 'reject')          AS rejected,
       COUNT_IF(t.ROUTE = 'needs_review')    AS to_review,
       COUNT_IF(t.ROUTE = 'auto_settle')     AS slipped_to_auto
FROM DEMO_CLM_GROUND_TRUTH gt
JOIN DT_DEMO_CLM_TRIAGED t ON t.CLAIM_NO = gt.CLAIM_NO
WHERE gt.PLANTED_FRAUD IS NOT NULL
GROUP BY gt.PLANTED_FRAUD
ORDER BY planted DESC;

**Extraction accuracy** -- the structured fields, against the planted truth (exact match on the claimant name and the claimed amount within a dollar).

In [ ]:
SELECT COUNT(*)                                                        AS claims,
       COUNT_IF(ABS(t.AMOUNT_CLAIMED - gt.AMOUNT_CLAIMED) < 1)           AS amount_exact,
       COUNT_IF(LOWER(TRIM(t.CLAIMANT)) = LOWER(TRIM(gt.CLAIMANT)))      AS claimant_exact,
       ROUND(100 * COUNT_IF(LOWER(TRIM(t.CLAIMANT)) = LOWER(TRIM(gt.CLAIMANT))) / COUNT(*), 1) AS claimant_pct
FROM DT_DEMO_CLM_TRIAGED t
JOIN DEMO_CLM_GROUND_TRUTH gt ON gt.CLAIM_NO = t.CLAIM_NO;

## Scale

This demo runs on **40 claims (~146 documents)**, but nothing about the pipeline is sized to that. It is `INCREMENTAL` end-to-end: a stream + task lands new files, and every dynamic table refreshes **only on the new rows** -- so each `AI_CLASSIFY`, `AI_PARSE_DOCUMENT`, `AI_EXTRACT`, and vision call runs **once per document, ever**. The same declarative SQL processes a daily inflow of thousands the same way it processes these 40; cost scales with *new documents*, not with the size of the book.

Adding a new document type is additive: extend the `AI_CLASSIFY` label set, add one extractor DT for the new type, and join it into the claim record -- no reprocessing of what's already landed.

> **Text-only variant:** drop `DT_DEMO_CLM_PHOTO` and the pipeline still runs on forms alone -- photos are classified, counted, and attached by `CLAIM_NO`, and triage conservatively escalates any high-value claim with unreviewed photos. (`source_structured_extraction.py --skip-photos` builds the matching corpus.)